# Inference & Evaluation — NL to Cypher Model

Evaluate fine-tuned model menggunakan OpenAI-compatible API (vLLM / local / cloud).

**Mengikuti pola dari `sft_inference.ipynb` reference.**

**Requirements:**
- Fine-tuned model served via vLLM atau OpenAI-compatible endpoint
- Neo4j running untuk execution testing
- Test data (validation CSV dari step 05a)

## Setup

In [ ]:
import os, sys, json
from dotenv import load_dotenv
from openai import OpenAI

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

load_dotenv(os.path.join(ROOT_DIR, '.env'), override=True)

# --- Model Server Config ---
# Option 1: vLLM local server
openai_api_base = "http://localhost:8000/v1"
# Option 2: remote vLLM
# openai_api_base = "https://your-vllm-server.com/v1"
# Option 3: OpenAI API
# openai_api_base = "https://api.openai.com/v1"

openai_api_key = os.getenv('OPENAI_API_KEY', 'not-needed')

print(f"API Base: {openai_api_base}")

In [ ]:
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

# List available models
try:
    models = client.models.list()
    print("Available models:")
    print("-" * 60)
    for m in models.data:
        print(f"  - {m.id}")
    print("-" * 60)
    print(f"Total: {len(models.data)}")
except Exception as e:
    print(f"Cannot connect to API: {e}")
    print("Make sure vLLM server is running!")

In [ ]:
# Select model
MODEL_NAME = "Qwen/Qwen3-4b"  # Replace with your fine-tuned model name

## Load test data

In [ ]:
import pandas as pd

TEST_DATA_PATH = os.path.join(ROOT_DIR, 'finetuning', 'query_model', 'data', 'validation_data.csv')

df_test = pd.read_csv(TEST_DATA_PATH)
print(f"Test samples: {len(df_test)}")
df_test.head()

### Prompt template

In [ ]:
from finetuning.query_model.generate_training_data import SYSTEM_INSTRUCTION

system_prompt = SYSTEM_INSTRUCTION

def format_user_message(context: str, question: str) -> str:
    return f"<INPUT>\n<CONTEXT>\n{context}\n</CONTEXT>\n<QUESTION>\n{question}\n</QUESTION>\n</INPUT>"

# Preview
print("System prompt (first 300 chars):")
print(system_prompt[:300])
print("\n...")

### Sanity check — single inference

In [ ]:
import time

# Test with first sample
sample = df_test.iloc[0]
user_msg = format_user_message(sample['context'], sample['question'])

start = time.time()
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ],
    temperature=0.1,
    max_tokens=512,
)
elapsed = (time.time() - start) * 1000

predicted = response.choices[0].message.content.strip()

print(f"Q: {sample['question']}")
print(f"\nExpected: {sample['response']}")
print(f"\nPredicted: {predicted}")
print(f"\nInference time: {elapsed:.0f}ms")

## Batch Evaluation

In [ ]:
from finetuning.query_model.evaluate import evaluate

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')

metrics = evaluate(
    test_data_path=TEST_DATA_PATH,
    api_base=openai_api_base,
    api_key=openai_api_key,
    model_name=MODEL_NAME,
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
)

In [ ]:
# Detailed results
details_path = os.path.join(ROOT_DIR, 'finetuning', 'query_model', 'data', 'evaluation_details.csv')
if os.path.exists(details_path):
    df_results = pd.read_csv(details_path)
    print(f"Detailed results: {len(df_results)} samples")
    print()
    
    # Show incorrect predictions
    incorrect = df_results[~df_results['syntax_valid']]
    if len(incorrect) > 0:
        print(f"\nInvalid syntax ({len(incorrect)} samples):")
        for _, row in incorrect.head(5).iterrows():
            print(f"  Q: {row['question'][:60]}")
            print(f"  Predicted: {row['predicted_cypher'][:80]}")
            print()
    else:
        print("All predictions have valid syntax!")

    df_results.head(10)

## Results Summary

In [ ]:
# Load saved metrics
metrics_path = os.path.join(ROOT_DIR, 'finetuning', 'query_model', 'data', 'evaluation_results.json')
with open(metrics_path, 'r') as f:
    metrics = json.load(f)

# DoD check
targets = {
    'syntax_validity': 0.95,
    'execution_success': 0.90,
    'result_accuracy': 0.80,
}

print("="*60)
print("EVALUATION RESULTS vs DoD TARGETS")
print("="*60)
for metric, target in targets.items():
    actual = metrics.get(metric, 0)
    status = 'PASS' if actual >= target else 'FAIL'
    print(f"  {metric:20s}: {actual:.1%} (target >= {target:.0%}) [{status}]")

print(f"\n  {'exact_match':20s}: {metrics.get('exact_match', 0):.1%}")
print(f"  {'avg_inference_ms':20s}: {metrics.get('avg_inference_ms', 0):.0f}ms")